In [42]:
%%capture
%pip install -U accelerate peft trl bitsandbytes transformers
%pip install evaluate sacrebleu rouge-score bert-score

In [ ]:
from huggingface_hub import login
login(token="")

# Load Model - Configure and Set Up

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
# Import Libraries
# Core
import os
import torch
import pandas as pd
import numpy as np

# Transformers & Tokenization
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from transformers.trainer_utils import set_seed
from transformers import GenerationConfig

# PEFT
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Transformer Reinforcement Learning
from trl import SFTTrainer

# Data
from datasets import Dataset, DatasetDict


set_seed(42) # ensure reproducibility

In [ ]:
# 4-bit config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True, #recently updated
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

model_name = "BioMistral/BioMistral-7B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # safe default for decoder-only

# Set padding side to right
tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"

gen_config = GenerationConfig(
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
# Base model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False
model.config.pretraining_tp = 1
model.generation_config = gen_config
model.gradient_checkpointing_enable()

In [ ]:
# Prepare for k-bit training (important for QLoRA)
model = prepare_model_for_kbit_training(model,
                                        use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})

# LoRA config
peft_config = LoraConfig(
    lora_alpha=16,                           # Scaling factor for LoRA
    lora_dropout=0.05,                       # Add slight dropout for regularization
    r=64,                                    # Rank of the LoRA update matrices
    bias="none",                             # No bias reparameterization
    task_type="CAUSAL_LM",                   # Task type: Causal Language Modeling
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],  # Target modules for LoRA
)

model = get_peft_model(model, peft_config)

In [ ]:
output_dir = "/content/drive/MyDrive/MIT807/biomistral_qlora/"
os.makedirs(output_dir, exist_ok=True)
# Training Arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,   # reduce if OOM; use grad_accumulation to enlarge effective batch size
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    max_steps=1000,                    # use epochs-based termination; set if you prefer step-based
    learning_rate=2e-5,              # safer starting point; reduce if unstable
    lr_scheduler_type="cosine",      #"linear", #"constant", #recently added
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="adamw_torch",        # requires bitsandbytes/paged optimizer in your env
    logging_strategy="steps",
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    bf16=(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    fp16=(not torch.cuda.is_bf16_supported() and torch.cuda.is_available()),
    gradient_checkpointing=True,
    report_to="none",
)

# Load Data

In [ ]:
train_prompt_style = """
You are a helpful, respectful, honest, and safe diabetes care assistant.
Your answers must be evidence-based, socially unbiased, positive, and emotionally considerate, and must not include harmful, racist, unethical, sexist, toxic, dangerous, or illegal content.
Your role is to:
- Educate medical practitioners, caregivers, and patients using recognized sources.
- Provide decision support (summarize guidelines/risk factors) but never diagnose.
- Map symptoms for referral, explain treatments simply, and recommend South Africa-tailored diet/exercise plans.
- Remind the patient that you are not a substitute for professional care.
- For emergencies, tell the user to seek immediate medical advice from their doctor.
- If a question is unclear or lacks factual coherence, explain why.
- If you don’t know, say so and avoid false information.

## Instruction
{instruction} {context}

## Question
{question} {options}

## Response
Answer: {response} {cot}
"""

In [ ]:
def formatting_prompts_func(examples):
    inputs = examples["question"]
    outputs = examples["answer"]
    instructions = examples.get("instruction", [None] * len(inputs)) # Get instructions with default None
    options = examples.get("options", [None] * len(inputs)) # Get options with default None
    contexts = examples.get("context", [None] * len(inputs)) # Get contexts with default None
    cots = examples.get("complex_cot", [None] * len(inputs)) # Get cot with default None
    long_answers = examples.get("long_answer", [None] * len(inputs)) # Get cot with default None

    texts = []
    for question, response, instruction, context, option, cot, long_answer in zip(inputs, outputs, instructions, contexts, options, cots, long_answers):
        if (instruction is None) or (instruction == "") or (instruction == 'nan') or (instruction == "None"):
            instruction = (
                "You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning."
                "Please answer the following medical question."
            )
        if (cot is None) or (cot == "None") or (cot == 'nan'):
            if (long_answer is None) or (long_answer == "None") or (long_answer == "nan"):
                cot = ""
            else:
                cot = f"\nExplanation: {long_answer}"
        else:
            cot = f"\nExplanation: {cot}"

        if (context == "nan") or (context == "None") or (context is None):
            context = ""
        else:
            context = f"\n### Context: {context}"

        if (option is None) or (option == "None") or (option == 'nan'):
            option = ""
        else:
            option = f"\n### Options: {option}"

        text = train_prompt_style.format(
            instruction=instruction,
            context=context or "",  # Use empty string if context is None
            question=question,
            cot=cot,
            options=option, # Use empty string if options is None
            response=response,
        )
        texts.append(text)

    return {"text": texts}

In [ ]:
df = pd.read_json("/content/drive/MyDrive/MIT807/data/silver/question_answers/training.json")
df = df[['instruction', 'question', 'answer', 'context', 'complex_cot', 'category', 'options', 'long_answer']]
df.info()
df.head(1)

In [ ]:
# Convert lists in 'Context' column to strings
df['context'] = df['context'].apply(lambda x: str(x) if isinstance(x, list) else x)

# Convert dictionaries in 'options' column to strings
df['options'] = df['options'].apply(lambda x: str(x) if isinstance(x, dict) else x)

dataset = Dataset.from_pandas(df)
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)
print(dataset["text"][1])

In [ ]:
print(dataset["text"][100])

In [ ]:
#split to test, evaluation and training
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset["validation"] = dataset["test"]
del dataset["test"]
#dataset["train"][0] # Removed this line
train_dataset = dataset["train"] # Added this line
eval_dataset = dataset["validation"] # Added this line

In [ ]:
dataset

In [ ]:
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
print(eval_dataset['text'][5])

# Train

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable} / {total} ({100*trainable/total:.4f}%)")

In [ ]:
from transformers import TrainerCallback
class SaveAtEpochEndCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        # this flag tells Trainer to save at the end of each epoch
        control.should_save = True
        return control

In [ ]:
# Initialize the Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset, # Corrected line
    data_collator=data_collator,
    eval_dataset=eval_dataset, # Added this line
    callbacks=[
        SaveAtEpochEndCallback
    ]
)

In [ ]:
# train
trainer.train(resume_from_checkpoint=True)